# Gravity Data: Load Selected Variables and Spatial Ranges

This notebook provides a comprehensive guide to selective gravitational field data loading and spatial filtering in Mera.jl. You'll learn advanced techniques for efficiently loading only the gravity data you need from large gravitational field simulations.

## Learning Objectives

- Master selective gravitational field variable loading for memory optimization
- Apply spatial filtering and region selection techniques for gravity analysis
- Work with different coordinate systems and units for gravitational fields
- Understand center-relative coordinate systems for gravitational phenomena
- Optimize gravity data loading for large simulations

## Quick Reference: Gravity Data Selection Functions

This section provides a comprehensive reference of Mera.jl functions for selective gravitational field data loading and spatial filtering.

### Variable Selection
```julia
# Load all variables (default behavior)
grav = getgravity(info)

# Select specific variables by name
grav = getgravity(info, vars=[:epot, :ax, :ay])       # Potential field and accelerations
grav = getgravity(info, vars=[:var1, :var2, :var3])   # Using variable numbers

# Select variables without keyword (order matters: info, variables)
grav = getgravity(info, [:epot, :ax])                 # Multiple variables
grav = getgravity(info, :epot)                        # Single variable

# Common gravity variable names and numbers
# :varn1 or :cpu  → CPU number (= -1)
# :var1 or :epot  → Gravitational potential field (φ)
# :var2 or :ax    → X-acceleration component
# :var3 or :ay    → Y-acceleration component
# :var4 or :az    → Z-acceleration component
```

### Spatial Range Selection
```julia
# RAMSES standard notation (domain: [0:1]³)
grav = getgravity(info, xrange=[0.2, 0.8],           # X-range filter
                        yrange=[0.2, 0.8],           # Y-range filter  
                        zrange=[0.4, 0.6])           # Z-range filter

# Center-relative coordinates (RAMSES units)
grav = getgravity(info, xrange=[-0.3, 0.3],          # Relative to center
                        yrange=[-0.3, 0.3],
                        zrange=[-0.1, 0.1],
                        center=[0.5, 0.5, 0.5])

# Physical units (e.g., kpc)
grav = getgravity(info, xrange=[2., 22.],             # Physical coordinates
                        yrange=[2., 22.],
                        zrange=[22., 26.],
                        range_unit=:kpc)

# Center-relative with physical units
grav = getgravity(info, xrange=[-16., 16.],           # Relative to center in kpc
                        yrange=[-16., 16.],
                        zrange=[-2., 2.],
                        center=[24., 24., 24.],
                        range_unit=:kpc)

# Box center shortcuts
grav = getgravity(info, center=[:boxcenter])         # All dimensions centered
grav = getgravity(info, center=[:bc])                # Short form
grav = getgravity(info, center=[:bc, 24., :bc])      # Mixed: center x,z; fixed y
```

### Performance Optimization
```julia
# Limit refinement levels for faster loading
grav = getgravity(info, lmax=8)                      # Maximum level 8

# Combined optimizations
grav = getgravity(info, [:epot, :ax],                # Select variables
                        lmax=10,                     # Limit levels
                        xrange=[-10., 10.],          # Spatial range
                        yrange=[-10., 10.],
                        zrange=[-2., 2.],
                        center=[:bc],                # Box center
                        range_unit=:kpc)             # Physical units
```

### Available Physical Units
```julia
# Check available units in simulation
viewfields(info.scale)

# Common length units
:m, :km, :cm, :mm, :μm, :Mpc, :kpc, :pc, :ly, :au, :Rsun
```

## Getting Started: Simulation Setup

Before exploring gravitational field data selection techniques, let's load our simulation and examine its properties. This establishes the foundation for all subsequent gravity data loading operations.

In [1]:
# Example-data root. Point this at your own simulation folder, or set the
# MERA_EXAMPLES environment variable; every path below is built from it.
MERA_EXAMPLES = get(ENV, "MERA_EXAMPLES", "/Volumes/FASTStorage/Simulations/Mera-Tests");

using Mera
info = getinfo(300, "$MERA_EXAMPLES/RAMSES/mw_L10");

[ Info: Precompiling Mera [02f895e8-fdb1-4346-8fe6-c721699f5126](cache misses: include_dependency fsize change (2), incompatible header (6), dep missing source (1), mismatched flags (1))


[ Info: Precompiling Mera [02f895e8-fdb1-4346-8fe6-c721699f5126] (cache misses: include_dependency fsize change (4), incompatible header (12), dep missing source (2), mismatched flags (2))



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up



*__   __ _______ ______   _______ 


|  |_|  |       |    _ | |   _   |
|       |    ___|   | || |  |_|  |
|       |   |___|   |_||_|       |
|       |    ___|    __  |       |
| ||_|| |   |___|   |  | |   _   |
|_|   |_|_______|___|  |_|__| |__|
Mera v1.8.0

[Mera]: 2026-08-05T15:55:54.492



Code: RAMSES


output [300] summary:
mtime: 2023-04-09T05:34:09
ctime: 2025-06-21T18:31:24.020
simulation time: 445.89 [Myr]
boxlen: 48.0 [kpc]
ncpu: 640
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 10 --> cellsize(s): 750.0 [pc] - 46.88 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  

7  --> (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :pressure, :scalar_00, :scalar_01)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Nstars:   5.445150e+05 
particle-variables: 

7  --> (:vx, :vy, :vz, :mass, :family, :tag, :birth)
particle-descriptor: (:position_x, :position_y, :position_z, :velocity_x, :velocity_y, :velocity_z, :mass, :identity, :levelp, :family, :tag, :birth_time)
-------------------------------------------------------
rt:            false
clumps:           false
-------------------------------------------------------
namelist-file: 

("&COOLING_PARAMS", "&SF_PARAMS", "&AMR_PARAMS", "&BOUNDARY_PARAMS", "&OUTPUT_PARAMS", "&POISSON_PARAMS", "&RUN_PARAMS", "&FEEDBACK_PARAMS", "&HYDRO_PARAMS", "&INIT_PARAMS", "&REFINE_PARAMS")
-------------------------------------------------------
timer-file:       true
compilation-file: false
makefile:         true
patchfile:        true



## Variable Selection Techniques

Understanding how to selectively load gravitational field variables is crucial for efficient memory usage and faster analysis. Mera provides flexible approaches to gravity variable selection, from loading everything to precise field component targeting.

### Understanding Gravitational Field Variable References

Mera provides access to gravitational field components through predefined variable names. Understanding these reference methods enables precise control over gravity data loading.

**Core Gravitational Field Variables:**

| Variable | Symbol Format | Number Format | Description |
|----------|---------------|---------------|-------------|
| CPU Number | `:cpu` | `:varn1` | Processor identification (= -1) |
| Gravitational Potential | `:epot` | `:var1` | Gravitational potential field |
| X-Acceleration | `:ax` | `:var2` | Acceleration component in x-direction |
| Y-Acceleration | `:ay` | `:var3` | Acceleration component in y-direction |
| Z-Acceleration | `:az` | `:var4` | Acceleration component in z-direction |

**Key Features:**
- Variable order is flexible in function calls
- Both symbolic (`:epot`) and numeric (`:var1`) formats supported
- Future updates will support descriptor file variable names
- Consistent naming across all Mera gravity functions
- Direct access to potential and acceleration components

### Loading All Variables (Default Behavior)

The simplest approach is to load all available gravitational field variables. This is the default behavior when no specific variables are requested.

In [2]:
grav = getgravity(info);

[Mera]: Get gravity data: 2026-08-05T15:55:57.071



Key vars=(:level, :cx, :cy, :cz)


Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

domain:


xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:


   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



...

Final data size: 28320979 cells, 4 variables
Creating Table from 28320979 cells with max 4 threads...
   Threading: 4 threads for 8 columns


   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads


   Creating IndexedTable with 8 columns...
✓ Table created in 5.356 seconds


Memory used for data table :1.6880627572536469

 GB
-------------------------------------------------------



In [3]:
grav.data

Table with 28320979 rows, 8 columns:
level  cx   cy   cz   epot       ax         ay         az
───────────────────────────────────────────────────────────────────
6      1    1    1    -0.105458  0.0713717  0.0713739  0.0714421
6      1    1    2    -0.106574  0.0736603  0.0736626  0.071396
6      1    1    3    -0.107689  0.0759945  0.0759969  0.0712471
6      1    1    4    -0.1088    0.0783709  0.0783733  0.0709879
6      1    1    5    -0.109906  0.0807857  0.0807883  0.0706111
6      1    1    6    -0.111006  0.0832346  0.0832372  0.0701094
6      1    1    7    -0.112097  0.0857126  0.0857152  0.0694754
6      1    1    8    -0.113176  0.0882139  0.0882167  0.068702
6      1    1    9    -0.114243  0.0907326  0.0907354  0.0677824
6      1    1    10   -0.115294  0.0932614  0.0932643  0.0667098
6      1    1    11   -0.116327  0.095793   0.095796   0.0654782
6      1    1    12   -0.117339  0.0983188  0.0983218  0.064082
⋮
10     814  493  514  -0.28418   -0.734355  0.0468811  -0.

### Selecting Multiple Variables

Mera provides multiple ways to select specific gravitational field components. You can use keyword arguments or positional arguments with flexible syntax.

In [4]:
grav_a = getgravity(info, vars=[:epot, :ax]); 

[Mera]: Get gravity data: 2026-08-05T15:56:21.230

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2) = (:epot, :ax) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|▏                                                 |  ETA: 0:00:50 (78.61 ms/it)

Processing files:   1%|▌                                                 |  ETA: 0:00:30 (46.92 ms/it)

Processing files:   2%|▊                                                 |  ETA: 0:00:27 (42.91 ms/it)

Processing files:   5%|██▍                                               |  ETA: 0:00:19 (31.13 ms/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:13 (21.58 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 2 variables
Creating Table from 28320979 cells with max 4 threads...
   Threading: 4 threads for 6 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads


   Creating IndexedTable with 6 columns...
✓ Table created in 2.381 seconds


Memory used for data table :1.2660471182316542 GB
-------------------------------------------------------



**Alternative:** Use variable numbers instead of symbolic names. This approach provides identical functionality with numeric references:

In [5]:
grav_a = getgravity(info, vars=[:var1, :var2]); 

[Mera]: Get gravity data: 2026-08-05T15:56:38.547

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2) = (:epot, :ax) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|▏                                                 |  ETA: 0:00:54 (84.61 ms/it)

Processing files:   1%|▌                                                 |  ETA: 0:00:31 (49.21 ms/it)

Processing files:   2%|▊                                                 |  ETA: 0:00:28 (43.75 ms/it)

Processing files:   3%|█▌                                                |  ETA: 0:00:21 (34.30 ms/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:13 (21.47 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 2 variables
Creating Table from 28320979 cells with max 4 threads...
   Threading: 4 threads for 6 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads


   Creating IndexedTable with 6 columns...
✓ Table created in 1.529 seconds


Memory used for data table :1.2660471182316542 GB
-------------------------------------------------------



**Keyword-free syntax:** When following the specific order (InfoType object, then variables), keyword arguments are optional:

In [6]:
grav_a = getgravity(info, [:epot, :ax]); 

[Mera]: Get gravity data: 2026-08-05T15:56:54.463

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2) = (:epot, :ax) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|▏                                                 |  ETA: 0:00:53 (83.56 ms/it)

Processing files:   1%|▌                                                 |  ETA: 0:00:30 (47.59 ms/it)

Processing files:   2%|▊                                                 |  ETA: 0:00:27 (42.81 ms/it)

Processing files:   3%|█▍                                                |  ETA: 0:00:22 (35.10 ms/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:13 (21.24 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 2 variables
Creating Table from 28320979 cells with max 4 threads...
   Threading: 4 threads for 6 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads


   Creating IndexedTable with 6 columns...
✓ Table created in 1.423 seconds


Memory used for data table :1.2660471182316542 GB
-------------------------------------------------------



In [7]:
grav_a.data

Table with 28320979 rows, 6 columns:
level  cx   cy   cz   epot       ax
──────────────────────────────────────────
6      1    1    1    -0.105458  0.0713717
6      1    1    2    -0.106574  0.0736603
6      1    1    3    -0.107689  0.0759945
6      1    1    4    -0.1088    0.0783709
6      1    1    5    -0.109906  0.0807857
6      1    1    6    -0.111006  0.0832346
6      1    1    7    -0.112097  0.0857126
6      1    1    8    -0.113176  0.0882139
6      1    1    9    -0.114243  0.0907326
6      1    1    10   -0.115294  0.0932614
6      1    1    11   -0.116327  0.095793
6      1    1    12   -0.117339  0.0983188
⋮
10     814  493  514  -0.28418   -0.734355
10     814  494  509  -0.284171  -0.733368
10     814  494  510  -0.284196  -0.73424
10     814  494  511  -0.284214  -0.734832
10     814  494  512  -0.284225  -0.735242
10     814  494  513  -0.284228  -0.73512
10     814  494  514  -0.284224  -0.734709
10     814  495  511  -0.284256  -0.735055
10     814  495  512  -0.

### Selecting Single Variables

For single variable selection, arrays and keywords are unnecessary. Maintain the order: InfoType object, then variable symbol:

In [8]:
grav_c = getgravity(info, :ax ); 

[Mera]: Get gravity data: 2026-08-05T15:57:09.875

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(2,) = (:ax,) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   1%|▍                                                 |  ETA: 0:00:39 (61.23 ms/it)

Processing files:   2%|█                                                 |  ETA: 0:00:25 (40.58 ms/it)

Processing files:   2%|█▎                                                |  ETA: 0:00:24 (38.79 ms/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:14 (22.00 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 1 variables
Creating Table from 28320979 cells with max 4 threads...
   Threading: 4 threads for 5 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads


   Creating IndexedTable with 5 columns...
✓ Table created in 1.674 seconds


Memory used for data table :1.0550392987206578 GB
-------------------------------------------------------



In [9]:
grav_c.data

Table with 28320979 rows, 5 columns:
level  cx   cy   cz   ax
───────────────────────────────
6      1    1    1    0.0713717
6      1    1    2    0.0736603
6      1    1    3    0.0759945
6      1    1    4    0.0783709
6      1    1    5    0.0807857
6      1    1    6    0.0832346
6      1    1    7    0.0857126
6      1    1    8    0.0882139
6      1    1    9    0.0907326
6      1    1    10   0.0932614
6      1    1    11   0.095793
6      1    1    12   0.0983188
⋮
10     814  493  514  -0.734355
10     814  494  509  -0.733368
10     814  494  510  -0.73424
10     814  494  511  -0.734832
10     814  494  512  -0.735242
10     814  494  513  -0.73512
10     814  494  514  -0.734709
10     814  495  511  -0.735055
10     814  495  512  -0.73541
10     814  496  511  -0.735248
10     814  496  512  -0.735572

## Spatial Range Selection Techniques

Spatial filtering is essential for focusing gravitational field analysis on specific regions of interest. Mera offers multiple coordinate systems and reference methods to accommodate different gravitational analysis needs.

**Available Coordinate Systems:**
- **RAMSES Standard:** Normalized domain [0:1]³ 
- **Center-Relative:** Coordinates relative to specified points
- **Physical Units:** Real astronomical units (kpc, pc, etc.)
- **Box-Centered:** Convenient shortcuts for simulation center

This flexibility allows precise gravitational field region selection for targeted analysis while optimizing memory usage and computational efficiency.

### RAMSES Standard Coordinate System

The RAMSES standard provides a normalized coordinate system that simplifies numerical calculations and ensures consistency across different simulation scales for gravitational field analysis.

**Coordinate System Properties:**
- **Domain Range:** [0:1]³ in all dimensions
- **Origin:** Located at [0., 0., 0.]
- **Benefits:** Scale-independent, numerically stable
- **Usage:** Ideal for relative positioning and field calculations

**Performance Optimization:** Use `lmax` to limit maximum refinement levels for faster loading and preview analysis. This is particularly useful for gravitational field analysis where you might not need the finest resolution everywhere.

In [10]:
grav = getgravity(info, lmax=8, 
                xrange=[0.2,0.8], 
                yrange=[0.2,0.8], 
                zrange=[0.4,0.6]); 

[Mera]: Get gravity data: 2026-08-05T15:57:26.233

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

domain:


xmin::xmax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
ymin::ymax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
zmin::zmax: 0.4 :: 0.6  	==> 19.2 [kpc] :: 28.8 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|▏                                                 |  ETA: 0:00:43 (68.03 ms/it)

Processing files:   2%|█                                                 |  ETA: 0:00:19 (30.37 ms/it)

Processing files:   3%|█▍                                                |  ETA: 0:00:17 (27.78 ms/it)

...

Processing files:  93%|██████████████████████████████████████████████▋   |  ETA: 0:00:00 (11.07 ms/it)

Processing files:  98%|█████████████████████████████████████████████████ |  ETA: 0:00:00 (10.84 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:06 (10.85 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 1233232 cells, 4 variables
Creating Table from 1233232 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.046 seconds
Memory used for data table :75.27139282226562

 MB
-------------------------------------------------------



**Range Verification:** The loaded gravitational field data ranges are stored in the `ranges` field using RAMSES standard notation (domain: [0:1]³):

In [11]:
grav.ranges

6-element Vector{Float64}:
 0.2
 0.8
 0.2
 0.8
 0.4
 0.6

### Center-Relative Coordinate Selection

Define spatial ranges relative to a specified center point. This approach is particularly useful for analyzing gravitational fields around specific massive objects, galaxies, or regions of interest:

In [12]:
grav = getgravity(info, lmax=8, 
                xrange=[-0.3, 0.3], 
                yrange=[-0.3, 0.3], 
                zrange=[-0.1, 0.1], 
                center=[0.5, 0.5, 0.5]); 

[Mera]: Get gravity data: 2026-08-05T15:57:33.657

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
ymin::ymax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
zmin::zmax: 0.4 :: 0.6  	==> 19.2 [kpc] :: 28.8 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   3%|█▌                                                |  ETA: 0:00:03 ( 5.05 ms/it)

Processing files:   7%|███▋                                              |  ETA: 0:00:03 ( 4.31 ms/it)

Processing files:  12%|█████▊                                            |  ETA: 0:00:03 ( 4.55 ms/it)

Processing files:  16%|███████▊                                          |  ETA: 0:00:02 ( 4.44 ms/it)

...

Processing files:  93%|██████████████████████████████████████████████▍   |  ETA: 0:00:00 ( 3.04 ms/it)

Processing files:  99%|█████████████████████████████████████████████████▌|  ETA: 0:00:00 ( 3.04 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:01 ( 3.07 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 1233232 cells, 4 variables
Creating Table from 1233232 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.047 seconds
Memory used for data table :75.27139282226562

 MB
-------------------------------------------------------



### Physical Unit Coordinate System

Working with physical units provides intuitive scale references for astronomical gravitational field analysis. This system automatically handles unit conversions and maintains physical meaning for gravitational phenomena.

**Key Advantages:**
- **Intuitive Scaling:** Use familiar astronomical units (kpc, pc, Mpc)
- **Automatic Conversion:** Mera handles unit transformations internally
- **Reference Point:** Coordinates measured from box corner [0., 0., 0.]
- **Flexibility:** Mix different units as needed for gravitational analysis

The following example demonstrates kiloparsec (kpc) coordinate selection for gravitational field analysis:

In [13]:
grav = getgravity(info, lmax=8, 
                xrange=[2.,22.], 
                yrange=[2.,22.], 
                zrange=[22.,26.], 
                range_unit=:kpc); 

[Mera]: Get gravity data: 2026-08-05T15:57:35.792

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

domain:
xmin::xmax: 0.0416667 :: 0.4583333  	==> 2.0 [kpc] :: 22.0 [kpc]
ymin::ymax: 0.0416667 :: 0.4583333  	==> 2.0 [kpc] :: 22.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   6%|███                                               |  ETA: 0:00:02 ( 2.81 ms/it)

Processing files:  12%|██████▏                                           |  ETA: 0:00:01 ( 2.64 ms/it)

Processing files:  19%|█████████▍                                        |  ETA: 0:00:01 ( 2.71 ms/it)

Processing files:  25%|████████████▍                                     |  ETA: 0:00:01 ( 2.67 ms/it)

...

Processing files:  94%|██████████████████████████████████████████████▉   |  ETA: 0:00:00 ( 2.69 ms/it)

Processing files:  99%|█████████████████████████████████████████████████▉|  ETA: 0:00:00 ( 2.80 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:01 ( 2.79 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 229992 cells, 4 variables
Creating Table from 229992 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.007 seconds
Memory used for data table :14.038482666015625

 MB
-------------------------------------------------------



**Available Physical Units:** The `range_unit` keyword accepts various length units defined in the simulation's `scale` field:

In [14]:
viewfields(info.scale)  # or e.g.: grav.info.scale


[Mera]: Fields to scale from user/code units to selected units
Mpc	= 0.0010000000000006482
kpc	= 1.0000000000006481
pc	= 1000.0000000006482
mpc	= 1.0000000000006482e6
ly	= 3261.5637769461323
Au	= 2.0626480623310105e23
km	= 3.0856775812820004e16
m	= 3.085677581282e19
cm	= 3.085677581282e21
mm	= 3.085677581282e22
μm	= 3.085677581282e25
Mpc3	= 1.0000000000019446e-9
kpc3	= 1.0000000000019444
pc3	= 1.0000000000019448e9
mpc3	= 1.0000000000019446e18
ly3	= 3.469585750743794e10
Au3	= 8.775571306099254e69
km3	= 2.9379989454983075e49
m3	= 2.9379989454983063e58
cm3	= 2.9379989454983065e64
mm3	= 2.937998945498306e67
μm3	= 2.937998945498306e76
Msol_pc3	= 0.9997234790001649
Msun_pc3	= 0.9997234790001649
g_cm3	= 6.76838218451376e-23
Msol_pc2	= 999.7234790008131
Msun_pc2	= 999.7234790008131
g_cm2	= 0.20885045168302602
Gyr	= 0.014910986463557083
Myr	= 14.910986463557084
yr	= 1.4910986463557083e7
s	= 4.70554946422349e14
ms	= 4.70554946422349e17
Msol	= 9.99723479002109e8
Msun	= 9.99723479002109e8
Mearth	

**Center-Relative with Physical Units:** Combine center-relative positioning with physical unit specifications for precise gravitational field analysis:

In [15]:
grav = getgravity(info, lmax=8, 
                xrange=[-16.,16.], 
                yrange=[-16.,16.], 
                zrange=[-2.,2.], 
                center=[24.,24.,24.], 
                range_unit=:kpc); 

[Mera]: Get gravity data: 2026-08-05T15:57:37.763

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   6%|██▉                                               |  ETA: 0:00:02 ( 2.73 ms/it)

Processing files:  12%|██████▏                                           |  ETA: 0:00:01 ( 2.61 ms/it)

Processing files:  18%|█████████▎                                        |  ETA: 0:00:01 ( 2.71 ms/it)

Processing files:  25%|████████████▌                                     |  ETA: 0:00:01 ( 2.63 ms/it)

...

Processing files:  94%|██████████████████████████████████████████████▉   |  ETA: 0:00:00 ( 2.64 ms/it)

Processing files:  99%|█████████████████████████████████████████████████▊|  ETA: 0:00:00 ( 2.65 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:01 ( 2.66 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 4 variables
Creating Table from 650848 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.024 seconds
Memory used for data table :39.725494384765625

 MB
-------------------------------------------------------



### Box Center Coordinate Shortcuts

Mera provides convenient shortcuts for box-centered coordinate systems, simplifying gravitational field analysis focused on the simulation center.

**Available Shortcuts:**
- `:bc` or `:boxcenter` - Center coordinate for all dimensions  
- Can be applied to individual dimensions selectively
- Combines seamlessly with physical units and range specifications
- Ideal for symmetric gravitational field analysis around simulation center

**Gravitational Field Benefits:**
- Perfect for studying gravitational effects around massive central objects
- Eliminates manual center calculation for field analysis
- Ensures precise geometric centering of gravitational field selections
- Simplifies symmetric region definitions for potential and acceleration studies
- Reduces coordinate specification errors in field filtering

In [16]:
grav = getgravity(info, lmax=8, 
                xrange=[-16., 16.], 
                yrange=[-16., 16.], 
                zrange=[-2., 2.], 
                center=[:boxcenter], 
                range_unit=:kpc); 

[Mera]: Get gravity data: 2026-08-05T15:57:39.597

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

center: [0.5, 0.5, 0.5] 

==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   6%|██▊                                               |  ETA: 0:00:02 ( 2.82 ms/it)

Processing files:  12%|██████                                            |  ETA: 0:00:02 ( 2.66 ms/it)

Processing files:  18%|█████████                                         |  ETA: 0:00:01 ( 2.75 ms/it)

...

Processing files:  91%|█████████████████████████████████████████████▌    |  ETA: 0:00:00 ( 2.72 ms/it)

Processing files:  97%|████████████████████████████████████████████████▌ |  ETA: 0:00:00 ( 2.72 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:01 ( 2.77 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 4 variables
Creating Table from 650848 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.021 seconds
Memory used for data table :39.725494384765625

 MB
-------------------------------------------------------



In [17]:
grav = getgravity(info, lmax=8, 
                xrange=[-16., 16.], 
                yrange=[-16., 16.], 
                zrange=[-2., 2.], 
                center=[:bc], 
                range_unit=:kpc); 

[Mera]: Get gravity data: 2026-08-05T15:57:41.564

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   6%|██▊                                               |  ETA: 0:00:02 ( 2.83 ms/it)

Processing files:  12%|██████                                            |  ETA: 0:00:02 ( 2.67 ms/it)

Processing files:  18%|█████████▏                                        |  ETA: 0:00:01 ( 2.63 ms/it)

Processing files:  24%|████████████▏                                     |  ETA: 0:00:01 ( 2.73 ms/it)

...

Processing files:  92%|█████████████████████████████████████████████▉    |  ETA: 0:00:00 ( 2.69 ms/it)

Processing files:  98%|█████████████████████████████████████████████████ |  ETA: 0:00:00 ( 2.72 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:01 ( 2.76 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 4 variables
Creating Table from 650848 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.026 seconds
Memory used for data table :39.725494384765625

 MB
-------------------------------------------------------



**Selective Dimension Centering:** Apply box center notation to specific dimensions while maintaining explicit coordinates for others. This example centers x and z dimensions while fixing y at 24 kpc:

In [18]:
grav = getgravity(info, lmax=8, 
                xrange=[-16., 16.], 
                yrange=[-16., 16.], 
                zrange=[-2., 2.], 
                center=[:bc, 24., :bc], 
                range_unit=:kpc); 

[Mera]: Get gravity data: 2026-08-05T15:57:43.464

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4) = (:epot, :ax, :ay, :az) 

center: [0.5, 0.5, 0.5] 

==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   6%|██▊                                               |  ETA: 0:00:02 ( 2.82 ms/it)

Processing files:  12%|██████                                            |  ETA: 0:00:02 ( 2.67 ms/it)

Processing files:  18%|█████████                                         |  ETA: 0:00:01 ( 2.63 ms/it)

...

Processing files:  92%|█████████████████████████████████████████████▉    |  ETA: 0:00:00 ( 2.74 ms/it)

Processing files:  98%|█████████████████████████████████████████████████ |  ETA: 0:00:00 ( 2.79 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:01 ( 2.82 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 4 variables
Creating Table from 650848 cells with max 4 threads...
   Threading: 4 threads for 8 columns
   Max threads requested: 4
   Available threads: 4
   Using parallel processing with 4 threads
   Creating IndexedTable with 8 columns...
✓ Table created in 0.023 seconds
Memory used for data table :39.725494384765625

 MB
-------------------------------------------------------



## Summary

This notebook demonstrated comprehensive gravitational field data selection techniques in Mera.jl, covering both variable selection and spatial filtering strategies for gravity data analysis. Key concepts covered include:

### Variable Selection Mastery
- **Flexible Reference Systems:** Using both symbolic (`:epot`) and numeric (`:var1`) variable references
- **Field Component Selection:** Choosing specific gravitational field components (potential vs. accelerations)
- **Selective Loading:** Optimizing memory usage by loading only required field variables
- **Syntax Variations:** Keyword and positional argument approaches for different coding styles
- **Single vs. Multiple Variables:** Appropriate syntax for different gravitational analysis scenarios

### Spatial Filtering Expertise  
- **Coordinate Systems:** RAMSES standard, physical units, center-relative, and box-centered approaches
- **Gravitational Focus:** Targeting regions with significant gravitational effects
- **Performance Optimization:** Using `lmax` restrictions and tight spatial bounds for field analysis
- **Unit Flexibility:** Working with various astronomical length scales for gravitational phenomena
- **Center Definitions:** Absolute positioning and relative coordinate systems for field studies

### Advanced Gravitational Techniques
- **Combined Selection:** Integrating variable selection with spatial filtering for gravity analysis
- **Memory Management:** Balancing analysis needs with computational resources for field calculations
- **Coordinate Shortcuts:** Using box center notation for simplified gravitational field positioning
- **Quality Assurance:** Verifying loaded field data ranges and component consistency
- **Multi-Physics Integration:** Preparing gravity data for combined hydro-gravity analysis